# Gene annotation and gene profiling



## The workflow:

*   Bacterial gene prediction: prodigal
*   Generating the gene catalog: cd-hit
*   Building a bowtie2 database from the gene catalog: bowtie2
*   Mapping short reads against the gene catalog: bowtie2
*   Calculating the coverage for each sample

## Setup of the environment



In [1]:
# conda environment
import os,sys
root_dir = "/biodata/resources/Gene_centric_analysis"

## Check the installation was successful

In [ ]:
! which prodigal
! which samtools
! which bowtie2
! which cd-hit


## Predict bacterial genes on contigs

In [ ]:
import os
# input folder
contigs_demo_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","contig_demo")
sample_id = "PSMB4MBK"
# input file: assembled contigs in fasta format
! ls -lh {contigs_demo_dir}/{sample_id}_contigs.fna.gz
# output folder
gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call")
! mkdir -p {gene_call_dir}

In [ ]:
# pre-calculated gene call results
gene_call_demo_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call_demo")

# check out the major output files
demo_output_gff = os.path.join(gene_call_demo_dir,sample_id+".prodigal.gff.gz")
demo_output_faa = os.path.join(gene_call_demo_dir,sample_id+".prodigal.faa.gz")
demo_output_fna = os.path.join(gene_call_demo_dir,sample_id+".prodigal.fna.gz")

! zcat {demo_output_gff} | head -n 5

In [ ]:
# define the major out put files
output_gff = os.path.join(gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(gene_call_dir,sample_id+".prodigal.fna")
# run gene prediction
! prodigal -p meta -i {contigs_demo_dir}/{sample_id}_contigs.fna.gz -f gff -o {output_gff} -a {output_faa} -d {output_fna}  > /dev/null 2>&1 # take ~2 min for run

## Overview of the output


In [ ]:
! zcat {demo_output_faa} | head -n 10

### Task: count all complete genes

Incomplete genes are marked by the "partial" field in the header:

*   A complete gene: **partial=00**
*   A gene that is incomplete on the left side: **partial=10**
*   A gene that is incomplete on the right side: **partial=01**
*   A gene that is incomplete on both sides: **partial=11**

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> Calculate the number of complete genes and the total number of genes.
</div>


Hint:

*   **grep** the headers with **partial=00**
*   count the results using the **wc** commend




<details>
<summary><strong>🔎 Solution :</strong></summary>

```

! grep "partial=00" {output_fna} | wc -l
! grep ">" {output_fna} | wc -l

```

</details>


### Put your solution here [2 min]


## Generating the gene catalog

A gene catalog consist of all possible, **non-redundent** protein-coding genes

Assuming that we have assembled genes from multiple samples, we are going to generate a gene catalog from these files.

We will start with a merged file with all the **complete** genes from multiple samples

In [ ]:
demo_gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog_demo")
merged_fa = os.path.join(demo_gene_catalog_dir,"merged_genes.fna.gz")
! zcat {merged_fa} | head -n 10

In [ ]:
coverage=90
identity=95
gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
! mkdir -p {gene_catalog_dir}
! cd-hit-est -i {merged_fa} -aS 0.{coverage} -aL 0.{coverage} -c 0.{identity} -M 0 -r 0 -B 0 -d 0 -o {gene_catalog_dir}/nr.fa -sc 1;

### Overview of the result

There are two major output files:


1.   The representative sequence of each cluster [fasta format]
2.   A cluster file recording the cluster each gene originates from [.clsr]



In [ ]:
! zcat {demo_gene_catalog_dir}/nr.fa.gz | head -n 10

In [ ]:
! zcat {demo_gene_catalog_dir}/nr.fa.clstr.gz | head -n 10

### Task: count the number of resulting clusters

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many clusters are there?
</div>

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

# option 1: 
! tail {demo_gene_catalog_dir}/nr.fa.clstr

# option 2:
! grep ">Cluster" {demo_gene_call_dir}/nr.fa.clstr | wc -l

```

</details>

## Create a bowtie2 database from the gene catalog



In [ ]:
# build bowtie2 database. 1 min for demo
! bowtie2-build {gene_catalog_dir}/nr.fa {gene_catalog_dir}/nr.fa_bowtie2DB

In [ ]:
! ls -lh {gene_catalog_dir}/nr.fa_bowtie2DB*

# Profile the abundance of each gene in each sample

**Workflow**


1.   Map the reads against the bowtie2 database
2.   Calculate the coverage of each gene





In [22]:
import os

gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
mgx_reads_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_reads")
output_sam_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_gf_mapping","raw")
! mkdir -p {output_sam_dir}

# define which sample you want to look at
sample_id="CSM5MCXT" 
# input reads
p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")


In [ ]:
! bowtie2 -x {gene_catalog_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset
! ls {gene_catalog_dir}

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many reads are mapped to the gene catalog? 
</div>

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many reads are mapped from sample PSMB4MBK? 
</div>
 

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

# define which sample you want to look at
sample_id="PSMB4MBK"

# input reads
p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")

! bowtie2 -x {gene_catalog_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset


```

</details>

### Profiling the coverage from the sam file

In [ ]:
! which samtools
! samtools --version
! samtools coverage --help

In [26]:
sample_id="CSM5MCXT" 
# sort the sam file
!samtools sort {output_sam_dir}/{sample_id}.sam -o {output_sam_dir}/{sample_id}.sorted_bam # 3 min
# generate the coverage file
!samtools coverage {output_sam_dir}/{sample_id}.sorted_bam > {output_sam_dir}/{sample_id}.coverage.txt

In [ ]:
# check the output:
! ls -lh {output_sam_dir}
! head {output_sam_dir}/{sample_id}.coverage.txt